# Loan Default Prediction

A probability-of-default (PD) model on the LendingClub dataset (2007-2018Q4).
The notebook keeps the exploratory analysis, the plots and the narrative; the
processing, training and evaluation steps live in the importable modules under
`src/` (see issue #4).

**How to run it:** install `requirements.txt`, place the raw file
`accepted_2007_to_2018Q4.csv` in `02_data/raw/`, then run the notebook from top
to bottom ("Restart & Run All"). Non-interactively that is:

```
jupyter nbconvert --to notebook --execute --inplace 01_notebooks/prediction.ipynb
```

The raw CSV (~1.6 GB) is not part of this repository; it comes from the
LendingClub dataset on Kaggle.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.calibration import CalibrationDisplay

# The notebook lives in 01_notebooks/, the src package in the project root.
sys.path.insert(0, "..")

from src.data_processing import (
    add_issue_year,
    create_target,
    drop_leakage_columns,
    filter_issue_years,
    load_raw_data,
    rm_nas,
)
from src.evaluate import evaluate_model
from src.features import engineer_features
from src.train import (
    save_model,
    split_and_scale,
    train_random_forest,
    train_scorecard,
)

sns.set_style("whitegrid")
warnings.filterwarnings('ignore')

## 1. Load Data

The raw file holds ~2.2M rows, so loading and processing it takes a few minutes
and a fair amount of memory. For a quick pass over the notebook, pass `nrows` to
`load_raw_data` to read only a sample.

In [ ]:
print("Loading the full dataset... This may take a few minutes.")
df = load_raw_data("../02_data/raw/accepted_2007_to_2018Q4.csv")

# For quick testing on a smaller sample:
# df = load_raw_data("../02_data/raw/accepted_2007_to_2018Q4.csv", nrows=100000)

print(f"Shape of the dataset: {df.shape}")

## 2. Exploratory Data Analysis

A first look at size, distributions and the correlation structure of the data.
The year plot motivates the one restriction this section already applies: the
filter to the 2015-2018 window a few cells below.

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df['grade'].value_counts()

In [ ]:
# Loan amounts are the exposure at risk, so their distribution sets the scale
# for everything that follows.
plt.figure(figsize=(10, 6))
sns.histplot(df['loan_amnt'], bins=50, kde=True)
plt.title('Distribution of Loan Amounts')
plt.show()

In [ ]:
# 'issue_year' is derived from 'issue_d'; it is used for the plot below and
# for the year filter in the next cell.
df = add_issue_year(df)

plt.figure(figsize=(12, 7))
sns.countplot(x='issue_year', data=df, palette='viridis')
plt.title('Number of Loans Issued Per Year', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Count of Loans', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Restrict the data to 2015-2018. The plot above shows that LendingClub's
# volume grew by orders of magnitude over the years, and lending standards
# changed with it. A recent, homogeneous window is closer to the population a
# scoring model would be applied to than the full 2007-2018 history.
print(f"Shape of the original DataFrame: {df.shape}")

df = filter_issue_years(df, start_year=2015, end_year=2018)

print(f"Shape of the new filtered DataFrame: {df.shape}")
print("\nYearly counts in the new filtered DataFrame:")
print(df['issue_year'].value_counts().sort_index())

In [ ]:
# Select only numeric columns for correlation
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(12, 10))
sns.heatmap(numeric_df.corr(), cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features')
plt.show()

## 3. Data Cleaning & Preprocessing

Four cleaning steps, in this order:

1. Keep only loans with a known outcome and derive the binary target from
   `loan_status`.
2. Drop columns that are mostly empty.
3. Drop columns that are irrelevant (identifiers, free text) or that leak the
   outcome.
4. Drop the rows that still have missing values.

The order matters: the target is defined first so that the later steps can be
judged against it, and the leakage drop runs after the missing-value column
drop (step 2) so that it also catches columns the 40% threshold happened to
keep.

In [ ]:
# Only 'Fully Paid' and 'Charged Off' loans have a final outcome. Loans that
# are still running would otherwise be labelled as non-defaults although their
# outcome is simply not known yet.
df = create_target(df)

print("Target variable 'target' created.")
print(df['target'].value_counts(normalize=True) * 100)

In [ ]:
# Drop columns with more than 40% missing values. Imputing a column that is
# mostly empty invents more data than it recovers; 40% is the point at which
# the LendingClub columns split cleanly into "generally populated" and
# "only filled for a special case" (hardship, settlement, joint applications).
df = rm_nas(df, threshold=0.4)
print(f"Shape after dropping columns with >40% NaNs: {df.shape}")

### Data leakage: post-outcome columns are removed

The first version of this notebook reported a ROC AUC of **0.9999**. On the
LendingClub dataset such a value is not a success but a known warning sign for
data leakage: several columns are only filled *after* the loan has already
defaulted or been repaid (payments received, outstanding principal, recoveries,
the continuously updated `last_fico_range_*` score, hardship and settlement
fields). Used as features, they tell the model the outcome it is supposed to
predict.

The cell below therefore removes all post-outcome columns in addition to the
irrelevant ones. What remains is the information that is available at
**origination** — exactly the information a real PD scoring model may use. The
resulting AUC is much lower and, for a credit default model on this kind of
data, the credible one (typically in the range of about 0.65-0.75).

In [ ]:
# Remove irrelevant explanatory variables (identifiers, free text, date
# columns) together with every post-outcome column described above. The
# concrete column lists live in src/data_processing.py.
df = drop_leakage_columns(df)
print(f"Shape after dropping irrelevant and post-outcome columns: {df.shape}")

In [ ]:
# After the column drops the remaining NaNs affect only a small share of rows,
# so dropping those rows is cheaper than imputing them.
df.dropna(inplace=True)
print(f"Shape after dropping all remaining NaN rows: {df.shape}")

## 4. Feature Engineering

Everything the model sees has to be numeric. `term` and `emp_length` are stored
as text but are genuinely ordinal, so they are parsed into numbers rather than
one-hot encoded. The remaining categorical columns get one-hot encoding with
`drop_first=True` to avoid the redundant reference category.

In [ ]:
# engineer_features copies its input, so binding the unencoded frame to a
# second name keeps it available for the WOE scorecard in section 7 instead
# of letting it be discarded here. No copy is made.
df_raw = df
df = engineer_features(df_raw)

print("Feature engineering complete.")
print(f"Final shape of data for modeling: {df.shape}")

## 5. Model Training & Evaluation

A Random Forest is used as the first model: it handles the mix of numeric and
one-hot encoded features without further preprocessing, copes with non-linear
relationships and is robust against the outliers that credit data is full of.
The interpretable counterpart that a bank would expect alongside it — logistic
regression on WOE-binned features — is built in section 7 and compared
against this model there.

Defaults are the minority class (roughly 20% of completed loans), so
`class_weight="balanced"` is used instead of resampling, and the train/test
split is stratified so both sets carry the same default rate.

In [ ]:
# The scaler inside split_and_scale is fitted on the training set only, so no
# information from the test set leaks into training.
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test, scaler = split_and_scale(
    X, y, test_size=0.3, random_state=42
)

print("Data split into training and test sets, and features scaled.")

In [ ]:
print("Training Random Forest model... This might be slow on the full dataset.")

model = train_random_forest(X_train, y_train)
save_model(model, "../04_models/random_forest.joblib")

print("Model training complete, model saved to 04_models/random_forest.joblib.")

### Discrimination: ROC AUC, Gini and KS

Three views of the same question - how well does the score separate the loans
that defaulted from the ones that were repaid:

- **ROC AUC** is the probability that a randomly drawn defaulted loan gets a
  higher score than a randomly drawn repaid one. Threshold-independent, so it
  says nothing about the 0.5 cutoff used in the classification report below.
- **Gini = 2 x AUC - 1** is the same information on a scale that maps a
  coin flip to 0 and a perfect ranking to 1. It is the form scorecard
  validation and model monitoring in banks are usually quoted in, which is why
  it is reported here even though it carries no information beyond the AUC.
- **KS** (Kolmogorov-Smirnov) is the largest gap between the cumulative score
  distributions of the two groups. Unlike AUC and Gini it is tied to a concrete
  point on the score scale: the cutoff at which the separation is greatest.

Rough orientation for retail PD models: Gini around 0.30-0.50 (AUC 0.65-0.75)
and KS around 0.20-0.35 are the usual range. Much higher values on this dataset
are a leakage warning rather than a success - see the section above.


In [ ]:
scores = evaluate_model(model, X_test, y_test)

print(f"ROC AUC: {scores['roc_auc']:.4f}")
print(f"Gini:    {scores['gini']:.4f}")
print(f"KS:      {scores['ks']:.4f}")
print("\nClassification Report:")
print(scores['report'])


### Calibration

Discrimination only checks the *ranking*. A PD model is also expected to get
the *level* right: a loan scored 0.10 should default about 10% of the time.
That matters as soon as the output feeds an expected-loss calculation - under
IFRS 9 the ECL is PD x LGD x EAD, so a PD that is systematically too high
inflates the provisions no matter how well the model ranks.

The reliability diagram below bins the predicted probabilities and plots the
observed default rate of each bin against the mean prediction in it. A
perfectly calibrated model follows the diagonal; points below it mean the model
predicts higher default rates than actually occur, points above it the reverse.

The model here is trained with `class_weight="balanced"`, which deliberately
tells it to treat defaults as if they were as frequent as non-defaults. That
improves the ranking of the minority class but pushes the predicted
probabilities above the true default rate, so the curve is expected to sit
below the diagonal. The fix is not to drop the class weights but to recalibrate
the scores afterwards (`CalibratedClassifierCV`) before using them as
probabilities.


In [ ]:
CalibrationDisplay.from_predictions(
    y_test, scores['y_pred_proba'], n_bins=10, name='Random Forest'
)
plt.title('Calibration Curve (Reliability Diagram)')
plt.show()


## 6. Model Interpretability

A credit decision has to be explainable: the applicant is entitled to a reason
for a rejection, and model validation has to be able to check that the model
keys on risk drivers that make economic sense rather than on an artefact of the
data. A random forest gives no such explanation by itself, so two views are
added here.

`feature_importances_` answers "which features does the forest use", SHAP
answers "in which direction, and by how much, does each feature move *this*
prediction". The second question is the one a credit risk function actually
asks.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)

plt.figure(figsize=(10, 7))
importances.nlargest(15).sort_values().plot.barh()
plt.title('Top 15 Features by Random Forest Importance')
plt.xlabel('Mean decrease in impurity')
plt.tight_layout()
plt.show()

The bar chart above ranks features by mean decrease in impurity, which has two
well-known limits: it says nothing about the *direction* of an effect, and it
systematically favours continuous and high-cardinality features over binary
ones, because those offer the trees more places to split. A one-hot column can
therefore look unimportant here while still mattering for individual decisions.

SHAP values fix both problems. They attribute each single prediction to its
features, so they carry a sign, and they are comparable across feature types.
Following the issue, they are computed on a random sample rather than the full
test set - `TreeExplainer` on a forest with this many rows and columns is slow,
and a few thousand rows are enough for a stable summary plot.

In [ ]:
X_shap = X_test.sample(min(3000, len(X_test)), random_state=42)

# Index 1 of the last axis selects the contributions to the default class.
shap_values = shap.TreeExplainer(model)(X_shap)[:, :, 1]

shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title('SHAP Summary: Impact on Predicted Default Probability')
plt.tight_layout()
plt.show()

### Reading the summary plot

Each dot is one loan. Its position on the x-axis is that feature's contribution
to that loan's default probability - right of zero pushes the prediction
towards default, left of zero away from it. The colour is the feature value, so
"red on the right" means high values of the feature increase the predicted risk.

One caveat specific to this notebook: the features were standardized before
training, so the colour scale refers to the scaled values. Standardization is
monotonic, so red still means a high value in the original units - but the
numbers on the colour bar are not dollars, percent or years.

Whether the pattern makes economic sense is the actual test, and three features
are worth checking explicitly:

- **`int_rate`** is expected to be the strongest single driver, with high rates
  pushing the prediction towards default. That is not leakage - the rate is set
  at origination - but it deserves a caveat rather than applause: LendingClub
  derives the rate from its own internal credit grade, and `grade`/`sub_grade`
  were dropped from the feature set as identifiers. A model dominated by
  `int_rate` is therefore mostly reproducing the platform's own underwriting
  decision instead of assessing the borrower independently. A bank building a
  scorecard would look at how much discriminatory power survives without it.
- **`dti`** (debt-to-income) is the most defensible driver economically: it
  measures how much of the borrower's income is already committed to debt
  service, and a borrower with little headroom has less capacity to absorb an
  income shock. Higher `dti` should push towards default; if it did not, the
  model would be suspect, not the theory.
- **`term`** should show 60-month loans as riskier than 36-month ones, for two
  reasons that compound: the loan is exposed to adverse events for longer, and
  borrowers who choose the longer term to get a lower instalment tend to be the
  more stretched ones to begin with. This is a good example of a feature whose
  effect is part duration and part self-selection.

In [ ]:
riskiest = int(model.predict_proba(X_shap)[:, 1].argmax())

shap.plots.waterfall(shap_values[riskiest], max_display=12, show=False)
plt.title('Single Case: Why This Loan Is Scored As High Risk')
plt.tight_layout()
plt.show()

The waterfall plot explains one single loan - the one the model scores as the
riskiest in the sample. It starts at the model's base value (`E[f(x)]`) and adds
each feature's contribution until it reaches the prediction for this loan
(`f(x)`).

`E[f(x)]` is not the average prediction over this sample. `TreeExplainer`
derives it from the training-data weights stored in the trees, and those weights
carry the `class_weight="balanced"` that `src/train.py` sets - so the baseline
sits near 0.5 rather than near the actual default rate of the portfolio. Every
contribution in the plot is measured against that balanced starting point, which
is the same caveat the calibration section makes: this model ranks, it does not
hand out PDs.

This is the format an adverse-action notice or a credit officer's override
needs: not "the model said 0.87", but which characteristics of this specific
application drove it there and by how much.


### Comments on Findings

The model above only uses features that are known at origination. The
corrected run on the full dataset is still outstanding (the raw CSV is not
part of this repository), so the concrete metrics are not filled in here yet.

What the corrected model must show is an AUC far below the 0.9999 of the first
version — that drop is the point: the earlier value came from post-outcome
columns leaking the target (see the section on data leakage above), not from
predictive power. A value in the region of 0.65-0.75 is what a realistic PD
model on LendingClub data achieves and is the number that can be defended in a
credit risk context - a Gini of roughly 0.30-0.50 in the form a scorecard
validation would report it.

For the business interpretation, recall on the 'Charged Off (1)' class is the
relevant quantity: it shows how many of the actually defaulting loans the model
flags. Precision and recall are now clearly traded off against each other, so
the decision threshold has to be chosen according to the cost of a missed
default versus a rejected good customer. The KS value above reports how large
that separation gets, not at which score it occurs, so it bounds what a
threshold can achieve rather than naming one.

The calibration curve has to be read separately from all of this: it says
nothing about ranking quality, only about whether the predicted probabilities
can be used as PDs in an expected-loss calculation. With
`class_weight="balanced"` they cannot be used unrecalibrated - which is
acceptable for a ranking model but has to be stated explicitly rather than
assumed away.

## 7. Benchmark: Logistic Regression Scorecard (WOE / IV)

The random forest is the modern model. The one a bank usually has to defend in
front of a supervisor is the older one: a **scorecard**, meaning logistic
regression on Weight-of-Evidence-transformed features. Building it here turns
the previous sections into a comparison rather than a single result.

**Weight of Evidence (WOE)** bins a feature and replaces every value by the
log-odds of its bin:

`WOE = ln(share of non-defaults in the bin / share of defaults in the bin)`

A bin that holds relatively many repaid loans gets a positive WOE, one that
holds relatively many defaults a negative one. The transformation is what makes
the approach acceptable to a model validation function:

- The relationship between feature and target no longer has to be linear — the
  bins absorb the shape, and the regression stays linear in the WOE values.
- Outliers cannot move a coefficient; they fall into an edge bin. Credit data
  is full of them (`annual_inc` in particular).
- Missing values are not imputed but get their own bin, so "not reported"
  becomes an explicit risk statement rather than a silent guess.
- Every score is traceable: the applicant landed in this bin, which carries
  this many points. That is exactly what an adverse-action notice needs.

**Information Value (IV)** aggregates the WOE of a feature into one number that
says how much it separates the two classes. The customary reading:

| IV | Reading |
| --- | --- |
| < 0.02 | barely predictive — drop |
| 0.02 - 0.1 | weak |
| 0.1 - 0.3 | medium |
| 0.3 - 0.5 | strong |
| > 0.5 | suspiciously strong — check for leakage before believing it |

The last row is the same instinct that drove section 3: a feature that
separates defaults almost perfectly is usually a feature that already knows the
outcome.

The binning itself is left to `optbinning`, which solves it as an optimisation
problem (maximise IV subject to a minimum bin size and monotonicity) rather
than cutting at fixed quantiles.

In [ ]:
# The scorecard bins the *unencoded* features. WOE grouping works on the raw
# categories, so the one-hot encoding from section 4 would only split them
# apart again; the ordinal parsing of 'term' and 'emp_length' is skipped for the
# same reason, since the binning groups their levels by default rate itself.
X_woe = df_raw.drop('target', axis=1)
categorical_variables = list(X_woe.select_dtypes(include=['str', 'object']).columns)

# split_and_scale keeps the original index, so selecting by it puts exactly the
# same loans in the same set for both models and the metrics stay comparable.
X_woe_train = X_woe.loc[X_train.index]
X_woe_test = X_woe.loc[X_test.index]

scorecard = train_scorecard(X_woe_train, y_train, categorical_variables)

n_numeric = X_woe.shape[1] - len(categorical_variables)
print(f"Scorecard trained on {len(categorical_variables)} categorical and "
      f"{n_numeric} numeric candidate features.")

### Information Value per feature

The table below is the feature selection, made explicit. `selected` is `False`
for every feature whose IV stayed under 0.02 — those are dropped before the
regression sees them, which is the classical way a scorecard keeps itself small
enough to be reviewed by hand.

In [ ]:
binning = scorecard.named_steps['woe']
iv_table = binning.summary()[['name', 'dtype', 'n_bins', 'iv', 'selected']]
iv_table = iv_table.sort_values('iv', ascending=False)

print(iv_table.to_string(index=False))

dropped = iv_table.loc[~iv_table['selected'], 'name'].tolist()
suspicious = iv_table.loc[iv_table['iv'] > 0.5, 'name'].tolist()

print(f"\nDropped, IV below 0.02: {dropped}")
print(f"Leakage candidates, IV above 0.5: {suspicious}")

Anything printed as a leakage candidate is a prompt to look, not a verdict.
Three things are worth checking in that table specifically:

- **`int_rate`** is the feature to expect at the top. LendingClub sets the rate
  from its own risk grade, so a high IV here is the platform's own credit
  assessment showing through rather than a post-outcome column that section 3
  missed. It is known at origination, so it stays — but a model leaning on it
  is partly reproducing someone else's scorecard, and a model document has to
  say so.
- **`fico_range_low` and `fico_range_high`** are the two ends of a four-point
  band, so they carry almost the same information and will report almost the
  same IV. IV is computed per feature and cannot see that; a production build
  would keep one of the pair and drop the other rather than hand the regression
  two copies of the same variable.
- **`issue_year`** is the open question left over from the leakage review: it is
  known at origination and so is not leakage, but as a time index it lets the
  model learn the vintage instead of the risk. The IV column is where that gets
  decided on evidence — if it clears the 0.02 threshold on the full dataset,
  the vintage is carrying signal and the feature deserves an explicit decision
  rather than a default one.

In [ ]:
scorecard_scores = evaluate_model(scorecard, X_woe_test, y_test)

comparison = pd.DataFrame(
    {
        'Random Forest': [scores['roc_auc'], scores['gini'], scores['ks']],
        'LR Scorecard (WOE)': [
            scorecard_scores['roc_auc'],
            scorecard_scores['gini'],
            scorecard_scores['ks'],
        ],
    },
    index=['ROC AUC', 'Gini', 'KS'],
)

print(comparison.round(4).to_string())
print("\nScorecard classification report:")
print(scorecard_scores['report'])

### Which model would be used in practice

Both models are evaluated on the same test rows with the same three metrics, so
the numbers above are directly comparable — and either of them may come out
ahead, which is the informative part.

**Reading the gap.** The forest can use interactions between features that the
scorecard, being additive in the WOE values, cannot represent at all, so a clear
win for the forest means there is non-linear structure the scorecard is missing.
The usual response is not to abandon the scorecard but to bin differently or to
add the interaction as its own feature. A scorecard that matches or beats the
forest says the opposite: the relationships in this data are close enough to
monotone that flexibility buys nothing, and the simpler model is simply the
better one.

**When the scorecard wins on grounds other than the metric.** For a regulatory
PD model — an IRB rating system under Basel, or the PD feeding an IFRS 9
expected-loss calculation — the model has to be explained, validated and
defended for years. The scorecard is additive and bin-wise, so every decision
decomposes into a handful of statements of the form "this applicant is in this
bin, which is worth this many points". Rejected applicants can be given a
reason; a validation unit can check each bin against its own expectation of the
risk direction; monitoring can watch bin populations drift over time. The
coefficients are few and stable, and the whole model fits on a page.

**When the forest wins.** Where the output is a ranking used internally and the
cost of a wrong decision is bounded — portfolio triage, collections
prioritisation, an early-warning list, or a challenger model testing whether the
scorecard leaves signal on the table — extra discrimination is worth more than
explainability, especially with the SHAP values from section 6 recovering a good
part of the latter.

**Why this is not simply accuracy versus interpretability.** Two of the
scorecard's properties are robustness rather than transparency: binning caps the
influence of outliers, and every feature has an explicit missing bin. The forest
handles neither by construction, and both matter more on credit data than the
metric difference usually does.

The honest summary is that a bank would build both: the scorecard is the model
of record, the forest is the challenger that keeps it honest.